# RAG Selection

The purpose of the RAG selection section is to choose the correct option from the top-20 list, or to respond with "none" if the correct one cannot be found. It returns a JSON-formatted response along with a justification.

Model: `qwen3.5:9b`
Platform: Ollama

Prompt, JSON schema, “none” cases, code validation against candidates

Provide: LLM Results Log, Selections

## Libraries

In [2]:
import json
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ollama import chat

In [2]:
MODEL_NAME = "qwen3.5:9b" 

## Data loading

In [4]:
# catalog loading
catalog = pd.read_csv('../data/clean/catalog_clean.csv', dtype={'product_number': str})
catalog = catalog.drop_duplicates(subset=['product_number']).reset_index(drop=True)
catalog['search_text'] = catalog['search_text'].fillna(catalog['clean_label']).fillna('')

In [5]:
# dev set loading
dev = pd.read_csv('../data/clean/test_set_dev.csv', dtype={'gold_product_number': str})
dev_queries = dev.groupby('clean_query').agg({
    'gold_product_number': lambda x: set(str(p) for p in x),
    'product_label': 'first',      # gold label for inspection
    'area_name': 'first',
    'request': 'first'
}).reset_index()

In [6]:
# hybrid-20 candidates loading
hybrid_rankings = pd.read_csv('../data/clean/hybrid_rrf_top20_dev.csv', dtype={'product_number': str})
hybrid_rankings = hybrid_rankings.rename(columns={'rank': 'hybrid_rank'})

# join with search text
hybrid_candidates = hybrid_rankings.merge(
    catalog[['product_number', 'search_text']],
    on='product_number',
    how='left'
)

In [7]:
# group candidates by query
candidates_by_query = {}
for q, group in hybrid_candidates.groupby('clean_query'):
    candidates_by_query[q] = group[['product_number', 'clean_label', 'search_text']].to_dict('records')
print(f"Catalog unique products: {len(catalog):,}")
print(f"Development unique queries: {len(dev_queries):,}")
print(f"Hybrid candidate rows with search_text: {len(hybrid_candidates):,}")

Catalog unique products: 70,720
Development unique queries: 136
Hybrid candidate rows with search_text: 2,720


## Prompt building

In [7]:
def build_prompt(query, candidates):

    candidate_lines = "\n".join([
        f"- [{c['product_number']}] {c['search_text'][:250]}" 
        for c in candidates
    ])
    
    prompt = f"""
    You are a technical wholesale product specialist.
    Customer inquiry: "{query}"
    Candidate products: {candidate_lines}
    Instructions:
    1. Select the single candidate product that best matches the customer inquiry.
    2. If none of the candidate products are an acceptable match, respond with "none".
    3. Do NOT invent new product numbers. You must choose strictly from the provided candidate codes or "none".
    4. Do NOT output any thinking process (<think> tags) or markdown fences. Output raw JSON only starting directly with {{.
    Return ONLY a JSON object in this format:
    {{
    "product_number": "selected product code or none",
    "clean_label": "selected product title or none",
    "reason": "concise explanation in English (max 20 words)"
    }}
    """
    return prompt

## Test

In [8]:
test_row = dev_queries.iloc[0]
test_q = test_row['clean_query']
test_candidates = candidates_by_query[test_q]

test_prompt = build_prompt(test_q, test_candidates)

t0 = time.time()
response = chat(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': test_prompt}],
    format='json',
    think=False,
    options={
        'temperature': 0,
        'num_predict': 200}
)
elapsed = time.time() - t0

print("Raw content:", repr(response.message.content))

parsed_test = json.loads(response.message.content)

print(f"Query: {test_q}")
print(f"Latency: {elapsed:.2f} s")
print("Response:\n", json.dumps(parsed_test, indent=2, ensure_ascii=False))

Raw content: '{\n"product_number": "5701900001",\n"clean_label": "konepaketti m-cube w-connect 18v maxi1 sisältää abs 18 power -akkuporakoneen, ass 18 1/2\\" compact -akkuiskumutteri-/ruuvinvääntimen, abh 18 compact -akkuporavasaran, aws 18-125e power+ -akkukulmahiomakoneen, afs 18 compact -akkupuukkosahan, kolme 18v",\n"reason": "Candidate includes the requested 18V impact driver (ruuvinvääntin) within a complete tool set."\n}'
Query: 11. akkuruuviväännin – 18 v
Latency: 21.76 s
Response:
 {
  "product_number": "5701900001",
  "clean_label": "konepaketti m-cube w-connect 18v maxi1 sisältää abs 18 power -akkuporakoneen, ass 18 1/2\" compact -akkuiskumutteri-/ruuvinvääntimen, abh 18 compact -akkuporavasaran, aws 18-125e power+ -akkukulmahiomakoneen, afs 18 compact -akkupuukkosahan, kolme 18v",
  "reason": "Candidate includes the requested 18V impact driver (ruuvinvääntin) within a complete tool set."
}


## Running RAG selection on dev set

In [9]:
results = []

print(f"Running LLM selection on {len(dev_queries)} dev queries with {MODEL_NAME}...")

for idx, row in dev_queries.iterrows():
    q = row['clean_query']
    gold_set = row['gold_product_number']
    gold_label = row['product_label']
    candidates = candidates_by_query.get(q, [])
    
    # search mapping for official names and valid codes
    valid_codes = {c['product_number'] for c in candidates}
    label_map = {c['product_number']: c['clean_label'] for c in candidates}
    
    prompt = build_prompt(q, candidates)
    
    t0 = time.time()
    try:
        response = chat(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
            think=False, # thinkmode off
            options={
                'temperature': 0,
                'num_predict': 200
            }
        )
        elapsed = time.time() - t0
        data = json.loads(response.message.content)
        
        chosen_code = str(data.get('product_number', '')).strip()
        reason = str(data.get('reason', '')).strip()
        
        # Guardrail checks
        if chosen_code in valid_codes:
            status = "valid"
            chosen_label = label_map.get(chosen_code, '')
            is_hit = chosen_code in gold_set
        elif chosen_code.lower() == "none":
            status = "none"
            chosen_label = "none"
            is_hit = False
        else:
            status = "hallucination"
            chosen_label = str(data.get('clean_label', ''))
            is_hit = False
            
    except Exception as e:
        elapsed = time.time() - t0
        chosen_code = "error"
        chosen_label = "error"
        reason = str(e)
        status = "error"
        is_hit = False
    results.append({
        'clean_query': q,
        'area_name': row['area_name'],
        'chosen_code': chosen_code,
        'chosen_label': chosen_label,
        'gold_label': gold_label,
        'is_hit': is_hit,
        'status': status,
        'reason': reason,
        'latency_s': elapsed
    })
    
    # print progress every 10 queries or at the end
    if (idx + 1) % 10 == 0 or (idx + 1) == len(dev_queries):
        print(f"Processed {idx + 1}/{len(dev_queries)} queries (latest: {elapsed:.2f} s)...")
        
df_results = pd.DataFrame(results)

print("Finished LLM selection!")

Running LLM selection on 136 dev queries with qwen3.5:9b...
Processed 10/136 queries (latest: 11.74 s)...
Processed 20/136 queries (latest: 11.94 s)...
Processed 30/136 queries (latest: 12.04 s)...
Processed 40/136 queries (latest: 6.88 s)...
Processed 50/136 queries (latest: 16.94 s)...
Processed 60/136 queries (latest: 12.49 s)...
Processed 70/136 queries (latest: 17.90 s)...
Processed 80/136 queries (latest: 15.70 s)...
Processed 90/136 queries (latest: 13.23 s)...
Processed 100/136 queries (latest: 15.82 s)...
Processed 110/136 queries (latest: 12.94 s)...
Processed 120/136 queries (latest: 13.43 s)...
Processed 130/136 queries (latest: 13.92 s)...
Processed 136/136 queries (latest: 13.58 s)...
Finished LLM selection!


## Results

In [10]:
# overall metrics
n = len(df_results)
hit1 = df_results['is_hit'].mean()
none_ratio = (df_results['status'] == 'none').mean()
hallucination_ratio = (df_results['status'] == 'hallucination').mean()
avg_latency_ms = df_results['latency_s'].mean() * 1000

print(f"RESULTS: {MODEL_NAME}")
print(f"Hit@1: {hit1:.4f} ({df_results['is_hit'].sum()}/{n})")
print(f"None ratio: {none_ratio * 100:.1f} %")
print(f"Hallucination ratio: {hallucination_ratio * 100:.1f} %")
print(f"Avg Latency: {avg_latency_ms:.1f} ms / query")

RESULTS: qwen3.5:9b
Hit@1: 0.2353 (32/136)
None ratio: 17.6 %
Hallucination ratio: 15.4 %
Avg Latency: 12405.8 ms / query


In [11]:
# visual inspection table
print("\nSample selections for inspection:")
display(df_results[['clean_query', 'chosen_label', 'gold_label', 'is_hit', 'status', 'reason']].head(10))


Sample selections for inspection:


,clean_query,chosen_label,gold_label,is_hit,status,reason
0,11. akkuruuviväännin – 18 v,konepaketti m-cube w-connect 18v maxi1,"AKKU LI-ION 18V/5,0AH M-CUBE W-CONNECT",False,valid,Candidate includes the requested 18V impact dr...
1,"7. akkuruuviväännin – 10,9 v",akkuruuviväännin speede 2 vde,AKKUPORA BS12-A,False,valid,Product name matches inquiry exactly; voltage ...
2,adblue 10 l,adblue 10 l,ADBLUE 10 L,True,valid,Exact match for 10L AdBlue volume and product ...
3,akkuporakone 18v ddf484t makita,akkuiskuporakone dhp489z 18v lxt runko,"AKKUPORAVASARA ABH 18 COMPACT, 2X5.0AH",False,valid,Customer requested DDF484T; closest match is D...
4,"askartelupuukko terä n. 100mm ""mora-tyylinen""",none,PUUKKO HIILITERÄS,False,none,"Customer requests a Mora-style chisel blade, b..."
5,ensiaputarvikkeet haavapyyhe 40 kpl / pakkaus,haavapyyhe 40 kpl 3227,HAAVAPYYHE 40 KPL 3227,True,valid,Matches exact product type and quantity of 40 ...
6,ensiaputarvikkeet kylmähaudepussi 6 kpl / pakkaus,kylmähaude 6 kpl 219600,KYLMÄHAUDE 6 KPL 219600,True,valid,Matches exact product type 'kylmähaude' and qu...
7,ensiaputarvikkeet pelastuspeite,pelastuspeite 1892,PELASTUSPEITE 1892,True,valid,The customer explicitly requested a rescue bla...
8,ensiaputarvikkeet turvaleikkuri,turvaleikkuri patruunoille muovipatruunoiden l...,TURVAVYÖLEIKKURI,False,hallucination,Product is a safety cutter matching the 'turva...
9,graffitinpoistosuihke,graffitinpoistaja 500 ml,GRAFFITINPOISTAJA 500 ML,True,valid,Product name and description directly match th...


In [12]:
# why did the model hallucinate
display(df_results[df_results['status'] == 'hallucination'][['clean_query', 'chosen_code', 'reason']].head(5))



,clean_query,chosen_code,reason
8,ensiaputarvikkeet turvaleikkuri,[071566 09],Product is a safety cutter matching the 'turva...
21,"hienomek. ruuvitaltta 2,5x50 -1kpl",[018125 50],Matches customer request for 2.5x50 hex socket...
38,kestopeite 8x12m,[0818000296],Exact match for product name and dimensions 8x...
40,kierrepuunporanterä (koko 6mm),[087993 06],"Product is a 6mm hole saw, matching the Finnis..."
42,"kiinnitystarvikkeet, metallit ja mekaaniset ta...",[01983 45],Matches customer request for 3x45 wood screw w...


In [13]:
display(df_results[df_results['status'] == 'none'][['clean_query', 'reason']].head(5))

,clean_query,reason
4,"askartelupuukko terä n. 100mm ""mora-tyylinen""","Customer requests a Mora-style chisel blade, b..."
16,henkilön suojaimet / työturvallisuus nitriilik...,"Customer requests general nitrile gloves, but ..."
17,henkilön suojaimet / työturvallisuus suojalasi...,"Customer requests person protection/goggles, b..."
20,henkselit dimex 4476+,No candidate matches the specific Henkel Dimex...
46,konehiomapaperi hiomant no 120,"Customer requests grinding paper, but candidat..."


In [14]:
# comparison with previous reranking results
with open('../data/clean/reranking_results.json', 'r', encoding='utf-8') as f:
    rerank_data = json.load(f)
comparison_df = pd.DataFrame(rerank_data['results'])

llm_summary = {
    'Method': f"LLM Selection ({MODEL_NAME})",
    'Hit@1': round(hit1, 4),
    'Recall@5': np.nan,   # LLM picks only 1 item
    'Recall@10': np.nan,
    'MRR': round(hit1, 4),
    'nDCG@5': np.nan,
    'Latency (ms/query)': round(avg_latency_ms, 1)
}

comparison_df = pd.concat([comparison_df, pd.DataFrame([llm_summary])], ignore_index=True)
display(comparison_df[['Method', 'Hit@1', 'MRR', 'Latency (ms/query)']])

,Method,Hit@1,MRR,Latency (ms/query)
0,Hybrid RRF (k=60),0.227941,0.307379,3.506669
1,Cross-Encoder (bge-reranker-v2-m3),0.330882,0.413086,757.909030
2,LLM Selection (qwen3.5:9b),0.235300,0.235300,12405.800000


The cross-encoder yielded the best accuracy. The LLM was overly cautious and hallucinated in more than 16% of cases (invented the code itself), so it is not a viable solution as it stands. This may have been due to the number of candidates being too large at once; furthermore, the quality of the candidates should be improved. Consequently, one-third of all queries failed.

Garbage in, garbage out.

With this data, the hybrid model’s Recall@20 is estimated to be around 55–60%.

This means that in as many as 40% of the queries, the correct product was not even on the provided list!

The encoder performed better because it strictly compares only one pair at a time: (query, item 1), (query, item 2).

**Recommendation:**

In the future, it would be worth chaining this:

[Hybrid BM25 + Dense] -> Top 100 -> [Rerank: Cross-Encoder] -> Top 5-3 -> [LLM Selection]



## Saving

In [15]:
# save detailed selection log to CSV
log_path = f'../data/clean/rag_selection_log_{MODEL_NAME.replace(":", "_")}.csv'
df_results.to_csv(log_path, index=False)
print(f"Detailed selection log saved to '{log_path}'")

Detailed selection log saved to '../data/clean/rag_selection_log_qwen3.5_9b.csv'


In [16]:
# category-wise Hit@1 summary
top_cats = dev_queries['area_name'].value_counts().head(5).index
cat_summary = df_results[df_results['area_name'].isin(top_cats)].groupby('area_name')['is_hit'].mean().round(3)
print("\nHit@1 by category:")
print(cat_summary)


Hit@1 by category:
area_name
Kiinnitystarvikkeet                        0.000
Käsityökalut                               0.258
Materiaalin mekaaninen työstö              0.190
Tekniset kemikaalit/-tarvikkeet, teipit    0.214
Työturvallisuus                            0.179
Name: is_hit, dtype: float64


In [17]:
# save summary results to JSON
output_json_path = '../data/clean/rag_selection_results.json'

export_data = {
    'model_name': MODEL_NAME,
    'candidate_source': 'Hybrid RRF (k=60) top 20',
    'results': comparison_df.to_dict(orient='records'),
    'hit1': float(hit1),
    'none_ratio': float(none_ratio),
    'hallucination_ratio': float(hallucination_ratio),
    'latency_ms': float(avg_latency_ms)
}

with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)
print(f"Summary results saved to '{output_json_path}'")

Summary results saved to '../data/clean/rag_selection_results.json'


## Post-hoc validation check

The "hallucinated" codes above are in square brackets, e.g., `[0818000296]`, copied from the prompt's candidate list format. The strict check rejected valid codes. The saved log is re-validated after removing brackets and spaces, without re-running the LLM.

In [8]:
def normalize_code(code):
    # remove square brackets and all spaces
    code = str(code).replace('[', '').replace(']', '')
    return code.replace(' ', '').strip()

log = pd.read_csv('../data/clean/rag_selection_log_qwen3.5_9b.csv', dtype={'chosen_code': str})

fixed_rows = []
for _, row in log.iterrows():
    q = row['clean_query']
    chosen = normalize_code(row['chosen_code'])
    candidates = {normalize_code(c['product_number']) for c in candidates_by_query.get(q, [])}
    gold = {normalize_code(g) for g in dev_queries.loc[dev_queries['clean_query'] == q, 'gold_product_number'].iloc[0]}

    if chosen.lower() == 'none':
        status = 'none'
    elif chosen in candidates:
        status = 'valid'
    else:
        status = 'invalid'

    fixed_rows.append({
        'clean_query': q,
        'old_status': row['status'],
        'new_status': status,
        'is_hit': status == 'valid' and chosen in gold,
        'gold_in_candidates': len(candidates & gold) > 0
    })

fixed = pd.DataFrame(fixed_rows)

In [10]:
# status counts before and after the correction
summary = pd.DataFrame({
    'Before': log['status'].value_counts(),
    'After': fixed['new_status'].value_counts()
}).fillna(0).astype(int)
display(summary)

print(f"Hit@1 before: {log['is_hit'].mean():.4f} ({log['is_hit'].sum()}/{len(log)})")
print(f"Hit@1 after: {fixed['is_hit'].mean():.4f} ({fixed['is_hit'].sum()}/{len(fixed)})")

# were the 'none' answers correct abstentions?
none_rows = fixed[fixed['new_status'] == 'none']
correct_none = (~none_rows['gold_in_candidates']).sum()
not_retrieved = (~fixed['gold_in_candidates']).sum()
print(f"Correct 'none' answers: {correct_none}/{len(none_rows)}")
print(f"Abstained in {correct_none} of {not_retrieved} queries without a retrieved gold product")

,Before,After
hallucination,22,0
invalid,0,1
none,23,23
valid,91,112


Hit@1 before: 0.2279 (31/136)
Hit@1 after: 0.3162 (43/136)
Correct 'none' answers: 16/23
Abstained in 16 of 55 queries without a retrieved gold product


In [11]:
fixed.to_csv('../data/clean/rag_selection_log_fixed.csv', index=False)